<a href="https://colab.research.google.com/github/elviakiran-miranda-hue/BUS4118S26/blob/dev/Prompt_ENG_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

class FinalTravelAgent:
    def __init__(self):
        # State Context (Must gather Name/Email before anything else)
        self.context = {"name": None, "email": None, "booking_id": None}
        self.escalated = False
        self.forbidden_patterns = [r'\d{3}-\d{2}-\d{4}', r'\d{9}']

    def internal_reasoning(self, user_input):
        """
        REASONING LOOP:
        1. Sanitize: Is there forbidden PII?
        2. Identify: Do I know who this is?
        3. Classify: Is this a booking or a complaint?
        4. Plan: What is the next missing piece of info?
        """

        # --- 1. Sanitize (Constraint) ---
        for p in self.forbidden_patterns:
            if re.search(p, user_input): return "BLOCK_PII", "Security Redaction triggered."

        # --- 2. Observe & Update Context ---
        # Email extraction
        email = re.search(r'\S+@\S+', user_input)
        if email: self.context['email'] = email.group(0)

        # Name extraction (Heuristic)
        if not self.context['name']:
            if "name is" in user_input.lower():
                self.context['name'] = user_input.lower().split("is")[-1].strip().title()
            elif len(user_input.split()) <= 3 and "@" not in user_input:
                self.context['name'] = user_input.strip().title()

        # Booking ID extraction
        bid = re.search(r'BK-\d{4}', user_input.upper())
        if bid: self.context['booking_id'] = bid.group(0)

        # --- 3. Determine Action ---
        # Action: Escalation (Legal/Medical)
        if any(w in user_input.lower() for w in ["lawyer", "sue", "emergency", "manager"]):
            self.escalated = True
            return "ESCALATE", "Connecting to human supervisor."

        # Action: Identity Gate
        if not self.context['name'] or not self.context['email']:
            return "GET_IDENTITY", "Need user identification."

        # Action: Support Flow (Cancel/Change)
        if any(w in user_input.lower() for w in ["cancel", "change", "issue", "problem"]):
            if not self.context['booking_id']:
                return "GET_BOOKING_ID", "Need booking reference for complaint."
            return "RESOLVE_COMPLAINT", "Processing cancellation/change."

        # Action: Sales Flow (Packages)
        if "package" in user_input.lower() or "book" in user_input.lower():
            return "START_BOOKING", "Offering new travel packages."

        return "GENERAL_GREETING", "Standard assistance."

    def execute_action(self, action_type, user_input):
        """The Execution phase based on the Reasoning Loop."""
        if action_type == "BLOCK_PII":
            return "I cannot process your request because it contains sensitive ID numbers. Please remove any Social Security info and try again."

        if action_type == "ESCALATE":
            return "I've flagged this for priority human review. A manager will join this chat shortly."

        if action_type == "GET_IDENTITY":
            return "Welcome! Before we start, I'll need your full name and email address to access our systems."

        if action_type == "GET_BOOKING_ID":
            return f"Thanks, {self.context['name']}. To help with your request, please provide your Booking Number (BK-XXXX)."

        if action_type == "RESOLVE_COMPLAINT":
            if "cancel" in user_input.lower():
                return f"I have successfully canceled booking {self.context['booking_id']}. A credit has been sent to {self.context['email']}."
            return f"I've opened the change request for {self.context['booking_id']}. What would you like to modify?"

        if action_type == "START_BOOKING":
            return f"Perfect, {self.context['name']}! I'm searching for the best packages for you now. Where are you dreaming of going?"

        return f"Hello {self.context['name']}, how can I help you with your travel plans today?"

    def start_chat(self):
        print("--- Travel Support AI Initialized ---")
        while not self.escalated:
            u_input = input("User: ")
            if not u_input.strip(): continue

            # REASON
            action_type, thought = self.internal_reasoning(u_input)

            # ACT
            response = self.execute_action(action_type, u_input)

            # FINAL ANSWER (Observation is fed back into the next loop)
            print(f"AI: {response}")

if __name__ == "__main__":
    bot = FinalTravelAgent()
    bot.start_chat()

--- Travel Support AI Initialized ---
User: Elvia
AI: Welcome! Before we start, I'll need your full name and email address to access our systems.
User: Elvia Miranda elvia@yahoo.com
AI: Hello Elvia, how can I help you with your travel plans today?
User: cancel a reservation
AI: Thanks, Elvia. To help with your request, please provide your Booking Number (BK-XXXX).
User: BK-200
AI: Hello Elvia, how can I help you with your travel plans today?
User: Cancel reservation
AI: Thanks, Elvia. To help with your request, please provide your Booking Number (BK-XXXX).
User: BK-2000
AI: Hello Elvia, how can I help you with your travel plans today?
User: cancel reserations
AI: I have successfully canceled booking BK-2000. A credit has been sent to elvia@yahoo.com.
